# `parse_pdf` + `apply_changes` — pattern extract / modify / rebuild côté PDF

Module testé : [`src/docpipeline/parsing/pdf/parse_pdf.py`](../src/docpipeline/parsing/pdf/parse_pdf.py).

## Le pattern transversal du projet

Symétrique de `parse_word` côté PDF :

```
extract  : parse_pdf(pdf_in)              → line_df avec span_id stable + bbox + style
modify   : on construit {span_id: nouveau_texte}
rebuild  : apply_changes(pdf_in, span_changes, pdf_out)
           → reconstruit le PDF en redact + insert_textbox sur les bbox d'origine
```

Le `span_id` (format `p_<page>_<line>`) est la clé stable, cohérente avec le `w_<para>_<run>` côté Word.

## Sortie de `parse_pdf(pdf)`

| Sortie | Contenu |
|---|---|
| `line_df`     | 1 ligne = 1 ligne de texte (`span_id`, `text`, bbox, font_name, font_size, **bold**, **italic**, **color**, is_invisible) |
| `image_df`    | 1 ligne = 1 image embarquée |
| `page_df`     | 1 ligne = 1 page (page_type, flags, extraction_strategy) |
| `doc_summary` | JSON synthèse |

## Limitations PDF (vs Word)

PyMuPDF ne permet de réinjecter du texte qu'avec les **Standard 14 fonts** (Helvetica, Times, Courier). Les polices custom du PDF d'origine ne peuvent pas être réembed-ées sans fournir le `.ttf`. Donc la fidélité visuelle est moindre qu'en Word, mais position / taille / couleur / gras / italique sont préservés.

## Démo : extract → uppercase → rebuild

On parse `tests/fixtures/notice_garanties.pdf`, on uppercase 5 spans, on reconstruit. Le PDF de sortie est sauvegardé dans `notebooks/_outputs/pdf_translation/` pour inspection visuelle.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.pdf.parse_pdf import parse_pdf, apply_changes

PDF_IN  = Path('../tests/fixtures/notice_garanties.pdf')
PDF_OUT = Path('_outputs/pdf_translation/notice_garanties_modified_js.pdf')
PDF_OUT.parent.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_colwidth', 120)

# 1) EXTRACT --------------------------------------------------------------
result = parse_pdf(PDF_IN)

display(Markdown('### 1. `doc_summary` — synthèse au niveau document'))
summary_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in result['doc_summary'].items()])
display(summary_df)

display(Markdown(f"### 2. `line_df` — {len(result['line_df'])} lignes extraites avec leurs styles"))
display(result['line_df'][['span_id', 'page_num', 'text', 'font_name', 'font_size', 'bold', 'italic', 'color']])

display(Markdown(f"### 3. `page_df` — {len(result['page_df'])} pages classifiées"))
display(result['page_df'][['page_num', 'page_type', 'tool', 'extraction_strategy', 'char_count', 'n_images']])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


### 1. `doc_summary` — synthèse au niveau document

,key,value
0,pdf_hash,dd01e383e6e578d16ee67cb42824fada2c0b935fccaadfb0875b2a715938e32a
1,file_size_bytes,2572
2,n_pages,1
3,source_category,other
4,source_tool,Unknown
5,source_confidence,0.3
6,source_signals,['no_metadata']
7,creator_raw,
8,producer_raw,
9,pdf_version,1.7


### 2. `line_df` — 9 lignes extraites avec leurs styles

,span_id,page_num,text,font_name,font_size,bold,italic,color
0,p_0_0,0,Notice d'Information · Garanties,Helvetica,18.0,False,False,#000000
1,p_0_1,0,Ce document présente les garanties souscrites dans le cadre du contrat.,Helvetica,11.0,False,False,#000000
2,p_0_2,0,La garantie IA (Individual Accident) couvre tout sinistre corporel.,Helvetica,11.0,False,False,#000000
3,p_0_3,0,La garantie BI (Business Interruption) couvre les pertes d'exploitation.,Helvetica,11.0,False,False,#000000
4,p_0_4,0,Tableau des garanties :,Helvetica,13.0,False,False,#000000
5,p_0_5,0,Garantie Plafond Franchise,Helvetica,10.0,False,False,#000000
6,p_0_6,0,IA · Individual Accident 50 000 EUR 300 EUR,Helvetica,10.0,False,False,#000000
7,p_0_7,0,BI · Business Interruption 100 000 EUR 1 000 EUR,Helvetica,10.0,False,False,#000000
8,p_0_8,0,RC · Responsabilite Civile 200 000 EUR 0 EUR,Helvetica,10.0,False,False,#000000


### 3. `page_df` — 1 pages classifiées

,page_num,page_type,tool,extraction_strategy,char_count,n_images
0,0,native,Unknown,native,483,0


In [3]:
# 2) MODIFY : uppercase les 5 premières lignes non vides
ld = result['line_df']
changes = {row['span_id']: row['text'].upper()
           for _, row in ld.head(5).iterrows()
           if row['text'].strip()}

display(Markdown(f"### 4. Modifications : {len(changes)} spans → uppercase"))
lines_idx = ld.set_index('span_id')
modifs_df = pd.DataFrame([
    {'span_id': sid, 'avant': lines_idx.loc[sid, 'text'], 'après': new}
    for sid, new in changes.items()
])
display(modifs_df)

# 3) REBUILD
res = apply_changes(PDF_IN, changes, PDF_OUT)
display(Markdown(f"**Fichier reconstruit** : `{PDF_OUT}` ({PDF_OUT.stat().st_size} bytes)"))
display(Markdown(f"- spans replaced : **{res['spans_replaced']}**"))
display(Markdown(f"- spans skipped : **{res['spans_skipped']}**"))
if res['warnings']:
    display(Markdown(f"- warnings : {res['warnings']}"))

### 4. Modifications : 5 spans → uppercase

,span_id,avant,après
0,p_0_0,Notice d'Information · Garanties,NOTICE D'INFORMATION · GARANTIES
1,p_0_1,Ce document présente les garanties souscrites dans le cadre du contrat.,CE DOCUMENT PRÉSENTE LES GARANTIES SOUSCRITES DANS LE CADRE DU CONTRAT.
2,p_0_2,La garantie IA (Individual Accident) couvre tout sinistre corporel.,LA GARANTIE IA (INDIVIDUAL ACCIDENT) COUVRE TOUT SINISTRE CORPOREL.
3,p_0_3,La garantie BI (Business Interruption) couvre les pertes d'exploitation.,LA GARANTIE BI (BUSINESS INTERRUPTION) COUVRE LES PERTES D'EXPLOITATION.
4,p_0_4,Tableau des garanties :,TABLEAU DES GARANTIES :


**Fichier reconstruit** : `_outputs\pdf_translation\notice_garanties_modified_js.pdf` (12854 bytes)

- spans replaced : **5**

- spans skipped : **0**

In [4]:
# 4) VERIF : on re-parse le PDF reconstruit et on compare
result2 = parse_pdf(PDF_OUT)
before = result['line_df']
after  = result2['line_df']

display(Markdown('### 5. Comparaison avant / après reconstruction (5 premières lignes)'))
n = min(5, len(before), len(after))
compare_df = pd.DataFrame({
    'span_id':       before['span_id'].head(n).values,
    'avant':         before['text'].head(n).values,
    'après':         after['text'].head(n).values,
})
display(compare_df)

display(Markdown('### 6. Bbox et font_size préservés ?'))
n_match = min(len(before), len(after))
checks = []
for col in ['bbox', 'font_size']:
    same = sum(1 for i in range(n_match) if str(before.iloc[i][col]) == str(after.iloc[i][col]))
    checks.append({'attribute': col, 'matches': f'{same} / {n_match}'})
display(pd.DataFrame(checks))

### 5. Comparaison avant / après reconstruction (5 premières lignes)

,span_id,avant,après
0,p_0_0,Notice d'Information · Garanties,Garantie Plafond Franchise
1,p_0_1,Ce document présente les garanties souscrites dans le cadre du contrat.,IA · Individual Accident 50 000 EUR 300 EUR
2,p_0_2,La garantie IA (Individual Accident) couvre tout sinistre corporel.,BI · Business Interruption 100 000 EUR 1 000 EUR
3,p_0_3,La garantie BI (Business Interruption) couvre les pertes d'exploitation.,RC · Responsabilite Civile 200 000 EUR 0 EUR
4,p_0_4,Tableau des garanties :,NOTICE D'INFORMATION · GARANTIES


### 6. Bbox et font_size préservés ?

,attribute,matches
0,bbox,0 / 9
1,font_size,0 / 9
